# Track A — Fase 0: Diagnosis OOF
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook ini menganalisis `oof.npy` dari Track B untuk:
- Menghitung per-class F1 + confusion matrix
- Menganalisis distribusi margin & entropy
- Mengecek error rate per fold
- Mengkonfirmasi klaim reviewer (Organic & Recyclable underfitting?)

> **Prasyarat:** `oof.npy` dari Track B sudah tersedia di Drive (Gate G1 terbuka)

---

In [ ]:
# ─── Cell 1: Mount Drive & Clone Repo ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run(
    ['git', 'clone', 'https://github.com/agaggigit/satria-data-bdcugm02.git',
     '/content/repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

import sys
sys.path.insert(0, '/content/repo')
print('✅ Repo di-clone dan path sudah ditambahkan')

In [ ]:
# ─── Cell 2: Install Dependencies ────────────────────────────────────────────
!pip install -q scipy seaborn scikit-learn

In [ ]:
# ─── Cell 3: Konfigurasi Path ─────────────────────────────────────────────────
# ⚠️ SESUAIKAN path di bawah ini dengan struktur Drive kamu!

DRIVE_BASE   = '/content/drive/MyDrive/BDC2026 apace'
OOF_PATH     = f'{DRIVE_BASE}/output_trackB/oof.npy'        # dari Track B
FOLDS_CSV    = f'{DRIVE_BASE}/output_trackA/folds.csv'      # dari Track A

# Output diagnosis
OUTPUT_DIR   = f'{DRIVE_BASE}/output_trackA/diagnosis'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'OOF path   : {OOF_PATH}')
print(f'folds.csv  : {FOLDS_CSV}')
print(f'Output dir : {OUTPUT_DIR}')

# Cek file ada
for path in [OOF_PATH, FOLDS_CSV]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'  ✅ {path} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ TIDAK DITEMUKAN: {path}')

In [ ]:
# ─── Cell 4: Load OOF ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

oof      = np.load(OOF_PATH)
folds_df = pd.read_csv(FOLDS_CSV)

print(f'OOF shape  : {oof.shape}')
print(f'folds rows : {len(folds_df):,}')
print(f'OOF prob range: [{oof.min():.4f}, {oof.max():.4f}]')
print(f'\nDistribusi label:')
print(folds_df['label'].value_counts().sort_index())

In [ ]:
# ─── Cell 5: Run Full Diagnosis ───────────────────────────────────────────────
from track_a.src.oof_diagnosis import run_full_diagnosis

results = run_full_diagnosis(
    oof_path   = OOF_PATH,
    folds_csv  = FOLDS_CSV,
    output_dir = OUTPUT_DIR,
    save_plots = True,
)

In [ ]:
# ─── Cell 6: Ringkasan untuk Tim ─────────────────────────────────────────────
metrics = results['metrics']
fold_df = results['fold_df']

print('=' * 60)
print('RINGKASAN UNTUK LAPORAN KE TIM:')
print('=' * 60)
print(f"\nCV Macro-F1 (dari oof_meta): 0.9421")
print(f"OOF Macro-F1 (argmax)      : {metrics['f1_macro']:.4f}")
print()
for name, f1 in zip(['Recyclable', 'Electronic', 'Organic'], metrics['f1_per_class']):
    print(f"  {name:15s}: F1 = {f1:.4f}")
print()
print('Error rate per fold:')
print(fold_df.to_string(index=False))

print()
print('→ Kirim hasil ini ke grup sebelum mulai Fase 1!')
print('  Terutama: apakah Organic & Recyclable memang terendah?')

In [ ]:
# ─── Cell 7 (OPSIONAL): Collect oof.npy dari Checkpoint ─────────────────────
# Jalankan ini HANYA jika oof.npy belum ada!
# Membutuhkan GPU (T4 atau lebih baik).

# Uncomment jika diperlukan:
# from track_b.src.oof_utils import collect_oof_from_checkpoints
#
# CHECKPOINT_DIR = f'{DRIVE_BASE}/output_trackB'
# oof_new = collect_oof_from_checkpoints(
#     checkpoint_dir = CHECKPOINT_DIR,
#     folds_csv      = FOLDS_CSV,
#     img_size       = 224,
#     batch_size     = 64,
# )
# np.save(OOF_PATH, oof_new)
# print(f'oof.npy disimpan ke {OOF_PATH}')

print('Sel ini opsional — skip jika oof.npy sudah ada.')

---

## ✅ Fase 0 Selesai!

Output yang dihasilkan di Drive:
- `diagnosis/oof_diagnosis.csv` — DataFrame lengkap (margin, entropy, prediksi per sampel)
- `diagnosis/oof_confusion_matrix.png`
- `diagnosis/oof_margin_entropy_dist.png`

**Next step:** Jalankan `04_cleanlab_cleaning.ipynb` untuk Fase 1 & 2.
